# Chapter 20 — Risk Assessment & Safety Factors
*Track 4: Engineers*

## 🎯 Learning Objectives
- Quantify risk as probability × consequence
- Design safety factors using probabilistic methods
- Build risk matrices and F-N curves

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
import pandas as pd

rng = np.random.default_rng(42)
plt.style.use("seaborn-v0_8-whitegrid")
print("Libraries loaded ✅")

## 1. Concept Review — Risk = Probability × Consequence

$$\text{Risk} = P(\text{hazard}) \times C(\text{consequence})$$

**Risk matrix** qualitatively ranks scenarios by:
- Likelihood (1–5 scale)
- Severity (1–5 scale)

**F-N curves** (Frequency-Number): cumulative frequency of events
causing N or more fatalities — used in nuclear/offshore industries.

In [ ]:
# Risk matrix
fig, ax = plt.subplots(figsize=(8, 6))
severity_labels   = ["Negligible", "Minor", "Moderate", "Major", "Catastrophic"]
likelihood_labels = ["Rare", "Unlikely", "Possible", "Likely", "Almost Certain"]

risk_colors = np.array([
    [1, 1, 2, 3, 3],
    [1, 2, 2, 3, 4],
    [1, 2, 3, 4, 4],
    [2, 3, 3, 4, 5],
    [2, 3, 4, 5, 5],
])[::-1]  # flip for display (low likelihood at bottom)

color_map = {1: "#2ecc71", 2: "#f1c40f", 3: "#e67e22", 4: "#e74c3c", 5: "#8e44ad"}
risk_labels = {1: "Low", 2: "Medium", 3: "High", 4: "Very High", 5: "Extreme"}

for i in range(5):
    for j in range(5):
        val = risk_colors[i, j]
        rect = plt.Rectangle([j, i], 1, 1, color=color_map[val], alpha=0.8)
        ax.add_patch(rect)
        ax.text(j+0.5, i+0.5, risk_labels[val], ha="center", va="center", fontsize=9, fontweight="bold")

ax.set_xticks(np.arange(5)+0.5); ax.set_xticklabels(severity_labels, fontsize=9)
ax.set_yticks(np.arange(5)+0.5); ax.set_yticklabels(likelihood_labels[::-1], fontsize=9)
ax.set_xlim(0,5); ax.set_ylim(0,5)
ax.set_xlabel("Severity →"); ax.set_ylabel("Likelihood →")
ax.set_title("5×5 Risk Matrix")
plt.tight_layout(); plt.show()

## 2. Math Walkthrough — Probabilistic Safety Factor

The **safety factor** $SF = R/S$. If $R$ and $S$ are lognormal:
$$\ln SF \sim N(\mu_{\ln R} - \mu_{\ln S},\; \sigma_{\ln R}^2 + \sigma_{\ln S}^2)$$

The probability of failure:
$$P_f = P(SF < 1) = \Phi\left(\frac{\mu_{\ln S} - \mu_{\ln R}}{\sqrt{\sigma_{\ln R}^2 + \sigma_{\ln S}^2}}\right)$$

In [ ]:
# Probabilistic safety factor for a structural connection
# Resistance R ~ Lognormal, Demand S ~ Lognormal
mu_R, cov_R = 100, 0.10   # kN
mu_S, cov_S =  60, 0.20   # kN

# Lognormal parameters
sig_lnR = np.sqrt(np.log(1 + cov_R**2))
mu_lnR  = np.log(mu_R) - 0.5*sig_lnR**2
sig_lnS = np.sqrt(np.log(1 + cov_S**2))
mu_lnS  = np.log(mu_S) - 0.5*sig_lnS**2

beta_lognormal = (mu_lnR - mu_lnS) / np.sqrt(sig_lnR**2 + sig_lnS**2)
pf = stats.norm.cdf(-beta_lognormal)
nominal_sf = mu_R / mu_S

print(f"Nominal safety factor: {nominal_sf:.2f}")
print(f"Reliability index β:   {beta_lognormal:.3f}")
print(f"Probability of failure: {pf:.6f}")
print(f"Return period:          {1/pf:.0f} cycles")

N = 100_000
R_mc = rng.lognormal(mu_lnR, sig_lnR, N)
S_mc = rng.lognormal(mu_lnS, sig_lnS, N)
print(f"
MC P(failure): {np.mean(R_mc < S_mc):.6f}")

In [ ]:
# F-N Curve
import matplotlib.ticker as ticker
fatalities = [1, 5, 10, 50, 100, 500, 1000]
freq       = [1e-2, 5e-3, 2e-3, 4e-4, 1e-4, 2e-5, 5e-6]  # per year

plt.figure(figsize=(8, 6))
plt.loglog(fatalities, freq, "ro-", lw=2, markersize=8)
# ALARP regions
n_range = np.array([1, 10000])
plt.fill_between(n_range, 1e-3/n_range, 1e-1/n_range, alpha=0.3, color="red", label="Intolerable")
plt.fill_between(n_range, 1e-5/n_range, 1e-3/n_range, alpha=0.2, color="yellow", label="ALARP")
plt.fill_between(n_range, 1e-7/n_range, 1e-5/n_range, alpha=0.2, color="green", label="Broadly acceptable")
plt.xlabel("Number of fatalities N"); plt.ylabel("Frequency F(N) [per year]")
plt.title("F-N Curve with ALARP Regions")
plt.legend(); plt.xlim(1, 1e4); plt.ylim(1e-8, 1e-1)
plt.tight_layout(); plt.show()